# Agente Orientador de Discentes do BTI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cleziojr/atividade_1_ia_agentica/blob/main/agente_orientador.ipynb)

### Como executar

**Colab:** adicione sua chave de api em *Secrets* (ícone de chave na barra lateral) e execute as células. A célula de setup clona o repositório e instala as dependências.

**VS Code:** crie um ambiente com Python 3.12, copie `.env.example` para `.env` com sua chave e selecione o kernel `.venv`:

```bash
uv venv --python 3.12 .venv
uv pip install -r requirements.txt
```

## IMPLEMENTAÇÃO

### CONFIGURAÇÃO E IMPORTAÇÃO DE MÓDULOS

In [1]:
import os
import sys

EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    if not os.path.exists("atividade_1_ia_agentica"):
        !git clone -q https://github.com/cleziojr/atividade_1_ia_agentica.git
    %cd atividade_1_ia_agentica
    %pip install -q -r requirements.txt

In [3]:
import os
import json
import urllib.request
import shutil
import subprocess
from pathlib import Path

from pydantic import BaseModel, Field

from agentkit.model import LLMAPI
from agentkit.agent import Agent
from agentkit.tools import tool, run_python

c:\Users\clezi\Documents\atividade_1_ia_agentica\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
if EM_COLAB:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    # Descomente o que for necessário
    # os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    # os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
else:
    from dotenv import load_dotenv

    load_dotenv()

In [5]:
# Modifique de acordo com o modelo que utilizará
llm = LLMAPI("gemini-3.8-flash", api_key=os.environ["GOOGLE_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/", temperature=0.0, max_tokens=4000)

In [14]:
SECURE_SYSTEM = """Avalie a estrutura e conteúdo da mensagem. Caso não mencione planejamento ou organização de disciplinas/matérias/componentes curriculares, encerre a conversa"""

## FERRAMENTAS

In [15]:
from typing import Literal
class IntencaoUsuario(BaseModel):
    intencao: Literal[
        "planejamento_matricula",
        "fora_escopo",
    ]
    justificativa: str

In [ ]:
from ferramentas_materias import comparar_materias_aluno

@tool
def recebe_input_usuario() -> str:
  """Permite que o usuário digite por meio da função input() nativa do Python. Retorna os últimos 300 caracteres de input do usuário.
  Exemplos de comandos de solicitação de informações para o usuário responder/comunicar: 
  'Descreva quais disciplinas você deseja cursar neste semestre. Informe suas prioridades e restrições de forma detalhada.'
  'Informe o que você acha que não funciona para você no planejamento de disciplinas que fiz para você, com o objetivo de eu melhorar'
  """
  input_usuario = input("")
  return input_usuario[:300]

@tool
def classifica_intencao_usuario(input_usuario: str) -> IntencaoUsuario:
    """Classifica se a solicitação está relacionada ao planejamento de disciplinas."""

    resposta = llm.generate_structured([
    {"role": "system", "content": SECURE_SYSTEM,},
    { "role": "user", "content": f""" 
    Analise a solicitação abaixo.
    Determine se a intenção do usuário está relacionada ao planejamento da grade de disciplinas do semestre.
    Solicitação: {input_usuario} """,
                },
            ],
            IntencaoUsuario,
            max_tokens=1000,
        )

    return resposta
  

@tool
def encerra_planejamento() -> None:
  """Encerra conversa conforme o LLM achar necessário"""
  sys.exit()
  return

# Descomente e adicione em TOOLS quando implementar (o @tool exige docstring)
# @tool
# def carrega_dados_disciplinas(): # -> list:
#   pass

# @tool
# def classifica_intencao_usuario():
#   pass

# @tool
# def recomenda_grade_disciplinas():
#   pass

# @tool
# def coleta_feedback():
#   pass

# @tool
# def calcula_carga_hora_disciplinas():
#   pass

SyntaxError: invalid syntax (65413545.py, line 23)

In [9]:
TOOLS = [comparar_materias_aluno, recebe_input_usuario]

### CLASSES E FUNÇÕES DE APOIO

#### PLANEJAMENTO

In [10]:
from agentkit.tools import render_tools

class Passo(BaseModel):
    instrucao: str
    resultado_esperado: str

class Planejamento(BaseModel):
  passos: list[Passo] = Field(min_length=2)

PLANNER_SYSTEM = f"""
Você escreve uma sequência de instruções e resultados esperados para um agente.
O agente só consegue agir por meio destas ferramentas:
{render_tools(TOOLS)}

Cada passo deve ser realizável com essas ferramentas ou com a conversa com o discente
(para obter informações que ele ainda não forneceu).
Ainda não escreva código.
"""

def cria_planejamento(tarefa: str) -> Planejamento:
  """Pede o plano ao modelo, com o papel no sistema e a tarefa no usuário."""
  response = llm.generate_structured(
        [{"role": "system", "content": PLANNER_SYSTEM}, {"role": "user", "content": tarefa}],
        Planejamento,
        max_tokens=4000,
    )
  return response

In [11]:
TAREFA = """
Elabore um planejamento de grade curricular para um discente que:

a) pode ou não saber quais disciplinas estão sendo ofertadas no semestre vigente;
b) pode ou não saber em quais disciplinas ele pode solicitar matrícula;

E que precisa saber quais:

c) disciplinas quer priorizar: componentes curriculares obrigatórios ou optativos;
d) são os dias e horários mais favoráveis para cursar disciplinas;
e) compromissos extracurriculares (estágio, projeto de pesquisa, esporte, grêmio, consultas médicas etc) bloqueiam dias e horários;
f) modalidades de disciplinas almeja priorizar (presencial, híbrido ou remoto?)
g) áreas de interesse almeja conhecer e/ou aprofundar: algoritmos e estruturas de dados? redes de computadores? ciência de dados? matemática pura? matemática aplicada?
h) docentes ofertam cada disciplina na grade proposta

E requer, ainda, que:

i) saiba qual é a carga horária almejada (quanto maior a carga horária, mais disciplinas, mais desafiante conciliar com outros compromissos e dedicar tempo para os estudos, mas não é impossível.)

---

O agente deve entregar o planejamento no tipo de arquivo e formato que o usuário solicitar, por exemplo: XML, YAML, Markdown etc.
Atente-se à linguagem do usuário, caso seja informal: pode ser .txt. Caso contrário, prepare um .pptx.
"""

planejamento = cria_planejamento(TAREFA)

for indice, passo in enumerate(planejamento.passos, start=1):
    print(f"{indice}. {passo.instrucao}")
    print(f"   espera: {passo.resultado_esperado}")

1. Solicitar ao discente a sua identificação ou matrícula (aluno_id) utilizando a ferramenta recebe_input_usuario() para viabilizar a consulta acadêmica.
   espera: Identificador do discente (aluno_id) devidamente coletado e disponível para as consultas subsequentes.
2. Executar a ferramenta comparar_materias_aluno(aluno_id) para levantar a lista completa de disciplinas ofertadas no semestre vigente e filtrar quais matérias o discente está apto a cursar, detalhando pré-requisitos cumpridos e eventuais impedimentos.
   espera: Mapeamento completo das disciplinas aptas para matrícula com respectivos códigos, nomes, docentes responsáveis, horários e modalidades ofertadas.
3. Utilizar a ferramenta recebe_input_usuario() para questionar o discente sobre suas preferências gerais: prioridade entre obrigatórias ou optativas, áreas de interesse para aprofundamento (ex: algoritmos, ciência de dados, redes, matemática pura/aplicada) e modalidade preferida (presencial, híbrida ou remota).
   esper

#### EXECUÇÃO

In [12]:
agent = Agent(llm, TOOLS, max_steps=10)

### DEFINIÇÃO DE TOOLS

In [13]:
EXECUTOR_SYSTEM = """Execute cada instrução existente no planejamento de modo utilizando as tools que achar adequadas"""

## RELATÓRIO

Sobre o uso de ReAct ou planejamento para a tarefa que propusemos que o agente cumpra (sugestão de grades curriculares para discentes), convém ressaltar que inicialmente a abordagem utilizando o planejamento aparenta ser mais atrativa, tendo em vista que permite que o agente não realize uma sequência de perguntas ao estudante até o próprio usuário saber que ainda não possui uma determinada informação. Um exemplo real desse cenário consiste em relação ao usuário manifestar profundo interesse em relação a uma determinada disciplina, mas antes do agente orientador elaborar próximos passos, questionar ao aluno se ele cumpre os pré-requsisitos exigidos para cursá-la. Ou seja, o agente já possui um grande contexto prévio, pois conhece a sequência de etapas necessárias para cumprir a tarefa que recebeu. No ReAct, isso não necessariamente ocorre dado que "[...] ele decide o próximo passo a cada rodada, a partir do resultado do passo anterior". Em suma, continua sendo uma abordagem válida, mas o fato do agente saber toda a sequência de passos a serem executados, para o contexto atual, aparenta ser mais estratégico (falta validar ainda).